In [1]:
import os
import time
import numpy as np
from PIL import Image

In [2]:
KERNEL = [
    [1/16, 2/16, 1/16],
    [2/16, 4/16, 2/16],
    [1/16, 2/16, 1/16]
]

In [3]:
def gaussian_blur_sequential(img):
    h, w = img.shape

    #zero padding
    padded = np.zeros((h + 2, w + 2), dtype=np.float32)
    padded[1:h + 1, 1:w + 1] = img

    out = np.zeros((h, w), dtype=np.float32)

    for i in range(h):
        for j in range(w):
            s = 0.0

            for ki in range(3):
                row = padded[i + ki]

                for kj in range(3):
                    s += KERNEL[ki][kj] * row[j + kj]

            out[i, j] = s

    return np.clip(out, 0, 255).astype(np.uint8)


def benchmark(image_path, repeats=3):
    img = np.array(Image.open(image_path).convert("L"), dtype=np.float32)

    h, w = img.shape
    times = []
    result = None

    print(f"  Image size: {w} x {h}")

    #Repeat filtering 
    for r in range(repeats):
        start = time.perf_counter()

        result = gaussian_blur_sequential(img)

        elapsed = time.perf_counter() - start
        times.append(elapsed)

        print(f"  Run {r + 1}: {elapsed:.4f} s")

    return result, times

In [4]:
if __name__ == "__main__":

    OUTPUT_DIR = "gaussian_outputs"
    REPEATS = 3

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    images = [
       "test_images/small_256x256.png",
        "test_images/medium_512x512.png",
        "test_images/large_1024x1024.png",
        "test_images/xlarge_2048x2048.png",
        "test_images/xxlarge_4096x4096.png"
    ]

    for path in images:
        if not os.path.exists(path):
            print(f"\nImage not found: {path}")
            continue

        filename = os.path.basename(path)
        print(f"\n{filename}:")

        result, times = benchmark(path, repeats=REPEATS)

        print(
            f"  Min = {min(times):.4f} s | "
            f"Avg = {sum(times) / len(times):.4f} s"
        )

        output_path = os.path.join(
            OUTPUT_DIR, f"gaussian_{filename}"
        )

        Image.fromarray(result).save(output_path)
        print(f"  Saved to: {output_path}")


small_256x256.png:
  Image size: 256 x 256
  Run 1: 1.6006 s
  Run 2: 1.6482 s
  Run 3: 1.6088 s
  Min = 1.6006 s | Avg = 1.6192 s
  Saved to: gaussian_outputs/gaussian_small_256x256.png

medium_512x512.png:
  Image size: 512 x 512
  Run 1: 6.4718 s
  Run 2: 6.3581 s
  Run 3: 6.3360 s
  Min = 6.3360 s | Avg = 6.3887 s
  Saved to: gaussian_outputs/gaussian_medium_512x512.png

large_1024x1024.png:
  Image size: 1024 x 1024
  Run 1: 25.7614 s
  Run 2: 25.8212 s
  Run 3: 25.9426 s
  Min = 25.7614 s | Avg = 25.8417 s
  Saved to: gaussian_outputs/gaussian_large_1024x1024.png

xlarge_2048x2048.png:
  Image size: 2048 x 2048
  Run 1: 102.7618 s
  Run 2: 101.4331 s
  Run 3: 101.3650 s
  Min = 101.3650 s | Avg = 101.8533 s
  Saved to: gaussian_outputs/gaussian_xlarge_2048x2048.png

xxlarge_4096x4096.png:
  Image size: 4096 x 4096


KeyboardInterrupt: 